# Local Disney Exploration (Pandas Only)

This notebook is for fast, local exploration of the Disney dataset without Spark dependencies.
It loads the Base and Enriched data using Pandas.

In [ ]:
import sys
from pathlib import Path
# Add src to sys.path so we can import stampli package
notebook_dir = Path("__file__").parent.resolve() if "__file__" in locals() else Path(".").resolve()
src_path = str(notebook_dir.parents[0]) # src/stampli -> src
if src_path not in sys.path:
    sys.path.append(src_path)
print(f"Added {src_path} to sys.path")


In [ ]:
from stampli.paths import get_enriched_path
ENRICHED_PATH = str(get_enriched_path())


In [ ]:
# 1. Load Data
print(f"Loading Base: {REVIEWS_PARQUET}...")
df_base = pd.read_parquet(REVIEWS_PARQUET)
print(f"Base Shape: {df_base.shape}")

print(f"Loading Enriched (Active): {REVIEWS_ENRICHED_PATH}...")
df_enriched = pd.read_parquet(REVIEWS_ENRICHED_PATH)
print(f"Enriched Shape: {df_enriched.shape}")

## Stage 1: Raw Ingestion Exploration
View the schema and initial rows of the raw Parquet output from Stage 1.

In [ ]:
print("Raw Ingestion Columns:", df_base.columns.tolist())
display_scrollable_dataframe(df_base.head(50))

## Stage 2 & 3: Enrichment Exploration
Analyzing the filtered and enriched dataset.

In [ ]:
# 2. Filter for Disney (Branch check)
print("Filter for 'Disney' branches...")
df_disney = df_enriched[df_enriched['Branch'].astype(str).str.lower().str.contains('disney', na=False)]
print(f"Disney Only Shape: {df_disney.shape}")
print("Branches Found:")
print(df_disney['Branch'].value_counts())


In [ ]:
# print df_disney size
len(df_disney)

In [ ]:
# 3. Explore Data
display_scrollable_dataframe(df_disney.head(100))

## Verification Blocks

In [ ]:
# Crowd Level Distribution (Extracted)
if 'crowd_level' in df_disney.columns:
    print("Crowd Level Distribution:")
    print(df_disney.groupby('Branch')['crowd_level'].value_counts(dropna=False))
else:
    print("Crowd Level not found.")

In [ ]:
# Keyword vs Extraction Check
KEYWORDS = ["crowd", "packed", "busy", "line", "queue", "wait", "full", "people"]

mask = df_disney['Review_Text'].astype(str).str.contains('|'.join(KEYWORDS), case=False, na=False)
subset = df_disney[mask]

print(f"Reviews with crowd keywords: {len(subset)}")
if 'crowd_level' in subset.columns:
    print("Extraction Status:")
    print(subset['crowd_level'].value_counts(dropna=False))

## Investigations (California June, etc)

In [ ]:
# Specific Investigation: California in June
def get_month(ym):
    try:
        return int(str(ym).split('-')[1])
    except:
        return 0

df_disney['month'] = df_disney['Year_Month'].apply(get_month)

ca_june = df_disney[
    (df_disney['Branch'] == 'Disneyland_California') & 
    (df_disney['month'] == 6)
]

print(f"Total Reviews (CA + June): {len(ca_june)}")
if 'crowd_level' in ca_june.columns:
    print("\nExtracted Crowd Levels:")
    print(ca_june['crowd_level'].value_counts(dropna=False))